# Baseline closed loop, one shot, stage by stage

One memory operation on one patch: d = 3, 9 QEC rounds, physical error rate 0.02 (noisy on purpose so defects show), 1.0 us cycle, reference link cards plus C2B, one LILLIPUT-class weak decoder (0.028 us per window), sliding windows commit 3 / buffer 3. Every stage below prints the data that exists at that stage in this run. Times are simulated microseconds.

Flow: planner -> controller -> QPU (rounds) -> controller (pulses to binary) -> packing -> link C2B -> Buffer 0 -> window manager -> decoder manager (assign unit) -> transfer into that unit's decoder memory (CWD) -> decoder engine (fetch, algorithm, release) -> result -> WDO -> Pauli frame; boundary handoff to the next window over DD at decode done.

In [1]:
import sys, os
REPO = os.path.abspath(os.getcwd() if os.path.isdir("decsim") else os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO); os.chdir(REPO)
from decsim.config import TICKS_PER_US
us = lambda ticks: round(ticks / TICKS_PER_US, 3)

def table(rows, columns):
    """Small aligned text table."""
    rows = [[str(r.get(c, "")) if isinstance(r, dict) else str(r[i]) for i, c in enumerate(columns)] for r in rows]
    widths = [max(len(c), *(len(r[i]) for r in rows)) if rows else len(c) for i, c in enumerate(columns)]
    line = lambda cells: "  ".join(cell.ljust(w) for cell, w in zip(cells, widths))
    print(line(columns)); print(line(["-" * w for w in widths]))
    for r in rows: print(line(r))

## 0. Tracing

Wrap the real methods so each stage records what passed through it, then run one shot of the baseline experiment configuration (`experiments/baseline_closed_loop.yaml`, 9 rounds).

In [2]:
from decsim.qpu import QPUDevice
from decsim.controller import Controller
from decsim.syndrome_ingress import SyndromeIngress
from decsim.syndrome_buffer import SyndromeBuffer
from decsim.window_manager import WindowManager
from decsim.decoder_manager import DecoderManager
from decsim.decoder_memory import DecoderMemory
from decsim.decoder_engine import DecoderEngine
from decsim.pauli_frame import PauliFrame

TRACE = []
CLOCK = {"engine": None}
def rec(stage, **data):
    TRACE.append(dict(t=us(CLOCK["engine"].now), stage=stage, **data))
def wrap(cls, name, before=None, after=None):
    original = getattr(cls, name)
    def wrapped(self, *args, **kwargs):
        if before: before(self, *args, **kwargs)
        result = original(self, *args, **kwargs)
        if after: after(self, result, *args, **kwargs)
        return result
    setattr(cls, name, wrapped)

def qpu_issue(self, command):
    CLOCK["engine"] = self.engine
    rec("controller -> qpu: issue", op=command.operation.name, rounds=command.round_count,
        cycle_us=us(command.round_ticks), starts_at=us(self.next_boundary()))
wrap(QPUDevice, "issue", before=qpu_issue)
def qpu_emit(self, payloads, operation):
    for p in payloads:
        rec("qpu: round emitted", op=operation.name, round=p.round_index, bits="".join(str(int(b)) for b in p.bits), size_bits=p.size_bits)
wrap(QPUDevice, "_emit", before=qpu_emit)
wrap(Controller, "accept_qpu_readout", before=lambda self, readout, route:
     rec("controller: readout accepted (pulses -> binary)", round=readout.round_index, binary_us=us(self.binary_availability_ticks)))
wrap(SyndromeIngress, "relay_syndrome", before=lambda self, payload, route:
     rec("packing: fragment relayed", round=payload.round_index, fragment=f"{payload.fragment_index + 1}/{payload.n_fragments}", size_bits=payload.size_bits))
def buffer_after(self, result, round_identity, **kw):
    s = self.snapshot()
    rec("buffer 0: round retained", round=round_identity[1], occupancy=s.occupancy, retained=[i[1] for i in s.retained_identities], bits="".join(map(str, result.fragments[0].bits)))
wrap(SyndromeBuffer, "finish_packing", after=buffer_after)
def wm_after(self, result, packet):
    w = [f"W{k}[{x.commit_lo}-{x.commit_hi}|buf->{x.buffer_hi}] " + ("READY" if x.t_data_complete else "waiting") for (o, k), x in sorted(self.windows.items())]
    rec("window manager: round arrived", round=packet.round_index, windows=" ".join(w))
wrap(WindowManager, "on_syndrome_arrival", after=wm_after)
wrap(DecoderManager, "enqueue", before=lambda self, job, reserve_transfer=None:
     rec("decoder manager: window enqueued", window=job.label, free_units={k: list(v) for k, v in self._free_units.items()}))
wrap(DecoderManager, "_start_job", after=lambda self, result, pool, job:
     rec("decoder manager: unit assigned", window=job.label, unit=job.unit, free_units={k: list(v) for k, v in self._free_units.items()}))
def deposit_after(self, result, job):
    rec("decoder memory: input landed in unit", unit=self.unit, window=job.label, rounds=[r.round_index for r in result.rounds], occupied_rounds=self.occupied_rounds)
wrap(DecoderMemory, "deposit", after=deposit_after)
wrap(DecoderManager, "_begin_service", before=lambda self, job: rec("decoder manager: start decode", window=job.label, unit=job.unit))
def decode_after(self, result, job):
    rec("decoder engine: result", window=job.label, syndrome_rounds_fetched=[f.round_index for f in job.payloads],
        defects=int(sum(sum(f.bits) for f in job.payloads)), correction_weight=int(sum(result.correction)) if result.correction is not None else None,
        logical=result.logical_observables, boundary_defects=result.boundary_defects)
wrap(DecoderEngine, "decode", after=decode_after)
wrap(WindowManager, "_send_boundary", before=lambda self, window, op, boundary, **kw:
     rec("boundary handoff (DD) at decode done", window=f"W{window.k}", to=[f"W{k}" for (_, k) in window.dependents], boundary=boundary))
wrap(PauliFrame, "commit_weak_correction", before=lambda self, **kw:
     rec("pauli frame: correction arrived (after WDO)", window=f"W{kw['window_key'][1]}", logical=kw["logical_observables"]))

from experiments.baseline_closed_loop import build_run, load_config
config = load_config("experiments/baseline_closed_loop.yaml")
config["rounds_per_shot"] = 9
config["noise_probability"] = 0.02      # noisier than the sweep so defects and corrections are visible
spec, decoder_engine = build_run(config, round_period_us=1.0, algorithm_latency_us=0.028, seed=0)
done = spec.build()
print("terminal status:", done.result.terminal_status, "| events traced:", len(TRACE))

terminal status: complete | events traced: 51


## 1. Planner: what the run was told before any data

The planner resolves each operation's round count and cadence and lays out the sliding windows in advance; the window manager holds them from the start, empty.

In [3]:
op = spec.ops[0]
resolved = done.controller._resolved_operations[op.id]
print("operation:", op.name, "| qubits", op.qubits, "| patches", op.patches, "| circuit rounds", resolved.round_count, "| cycle", us(resolved.round_ticks), "us")
print("code geometry:", resolved.code_geometry)
table([dict(window=f"W{k}", commit=f"{w.commit_lo}-{w.commit_hi}", buffer_through=w.buffer_hi, rounds_read=w.n_rounds, waits_on=[f"W{d[1]}" for d in w.deps])
       for (o, k), w in sorted(done.window_manager.windows.items())], ["window", "commit", "buffer_through", "rounds_read", "waits_on"])

operation: memory | qubits (0,) | patches (0,) | circuit rounds 9 | cycle 1.0 us
code geometry: ResolvedCodeGeometry(code_name='rotated surface code (d=3)', distance=3, commit_round_count=3, buffer_round_count=3, minimum_leading_buffer_round_count=3, minimum_trailing_buffer_round_count=3, one_patch_spatial_node_count=9, buffer_floor_override_active=False)
window  commit  buffer_through  rounds_read  waits_on
------  ------  --------------  -----------  --------
W0      1-3     6               6            []      
W1      4-9     9               6            ['W0']  


## 2. Controller -> QPU: the command, and the QPU's cycle clock

In [4]:
table([r for r in TRACE if r["stage"] == "controller -> qpu: issue"], ["t", "op", "rounds", "cycle_us", "starts_at"])

t    op      rounds  cycle_us  starts_at
---  ------  ------  --------  ---------
0.0  memory  9       1.0       0.0      


## 3. QPU: one syndrome round per cycle

Each round is the detector bits of that cycle from the Stim circuit (real sampled data), tagged (operation, patch, round).

In [5]:
table([r for r in TRACE if r["stage"] == "qpu: round emitted"], ["t", "op", "round", "bits", "size_bits"])

t    op      round  bits          size_bits
---  ------  -----  ------------  ---------
1.0  memory  1      0000          4        
2.0  memory  2      10110000      8        
3.0  memory  3      10100000      8        
4.0  memory  4      00000000      8        
5.0  memory  5      00000000      8        
6.0  memory  6      10000000      8        
7.0  memory  7      10000000      8        
8.0  memory  8      01001000      8        
9.0  memory  9      100100000001  12       


## 4. Controller (pulses -> binary), packing, link C2B, Buffer 0

The controller accepts the readout after the QC link and its binary-availability time (0 here); packing relays each fragment (one fragment per round in this configuration); the round crosses C2B into Buffer 0, which retains it. The buffer column shows what Buffer 0 holds after each round.

In [6]:
table([r for r in TRACE if r["stage"] in ("controller: readout accepted (pulses -> binary)", "packing: fragment relayed")], ["t", "stage", "round", "binary_us", "fragment", "size_bits"])
print()
c2b = [x for x in done.result.link_traffic["transfers"] if x["path"] == "c2b"]
table([dict(round=x["attribution"]["round_lo"], sent=us(x["send_ticks"]), delivered=us(x["delivery_ticks"]), bits=x["payload_bits"], delay_us=us(x["total_delay_ticks"])) for x in c2b], ["round", "sent", "delivered", "bits", "delay_us"])
print()
table([r for r in TRACE if r["stage"] == "buffer 0: round retained"], ["t", "round", "bits", "occupancy", "retained"])

t    stage                                            round  binary_us  fragment  size_bits
---  -----------------------------------------------  -----  ---------  --------  ---------
1.0  controller: readout accepted (pulses -> binary)  1      0.0                           
2.0  controller: readout accepted (pulses -> binary)  2      0.0                           
3.0  controller: readout accepted (pulses -> binary)  3      0.0                           
4.0  controller: readout accepted (pulses -> binary)  4      0.0                           
5.0  controller: readout accepted (pulses -> binary)  5      0.0                           
6.0  controller: readout accepted (pulses -> binary)  6      0.0                           
7.0  controller: readout accepted (pulses -> binary)  7      0.0                           
8.0  controller: readout accepted (pulses -> binary)  8      0.0                           
9.0  controller: readout accepted (pulses -> binary)  9      0.0                

## 5. Window manager: which windows are complete after each round

In [7]:
table([r for r in TRACE if r["stage"] == "window manager: round arrived"], ["t", "round", "windows"])

t      round  windows                                      
-----  -----  ---------------------------------------------
1.254  1      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
2.258  2      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
3.258  3      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
4.258  4      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
5.258  5      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
6.258  6      W0[1-3|buf->6] READY W1[4-9|buf->9] waiting  
7.258  7      W0[1-3|buf->6] READY W1[4-9|buf->9] waiting  
8.258  8      W0[1-3|buf->6] READY W1[4-9|buf->9] waiting  
9.262  9      W0[1-3|buf->6] READY W1[4-9|buf->9] READY    


## 6. Decoder manager: enqueue, free units, unit assignment

In [8]:
table([r for r in TRACE if r["stage"] in ("decoder manager: window enqueued", "decoder manager: unit assigned")], ["t", "stage", "window", "unit", "free_units"])

t      stage                             window                  unit  free_units      
-----  --------------------------------  ----------------------  ----  ----------------
6.258  decoder manager: window enqueued  memory W0 [commit 1-3]        {'default': [0]}
6.258  decoder manager: unit assigned    memory W0 [commit 1-3]  0     {'default': []} 
9.262  decoder manager: window enqueued  memory W1 [commit 4-9]        {'default': [0]}
9.262  decoder manager: unit assigned    memory W1 [commit 4-9]  0     {'default': []} 


## 7. Transfer into the assigned unit's decoder memory (link CWD), then the memory holds the input

In [9]:
cwd = [x for x in done.result.link_traffic["transfers"] if x["path"] == "cwd"]
table([dict(window=f"W{x['attribution']['window_id']}", rounds=f"{x['attribution']['round_lo']}-{x['attribution']['round_hi']}", sent=us(x["send_ticks"]), delivered=us(x["delivery_ticks"]), bits=x["payload_bits"], delay_us=us(x["total_delay_ticks"])) for x in cwd], ["window", "rounds", "sent", "delivered", "bits", "delay_us"])
print()
table([r for r in TRACE if r["stage"] == "decoder memory: input landed in unit"], ["t", "unit", "window", "rounds", "occupied_rounds"])
print()
print("decoder memory now:", [m.snapshot() for m in done.decoder_manager.decoder_memories.values()])

window  rounds  sent   delivered  bits  delay_us
------  ------  -----  ---------  ----  --------
W0      1-6     6.258  8.258      44    2.0     
W1      4-9     9.262  11.262     52    2.0     

t       unit  window                  rounds              occupied_rounds
------  ----  ----------------------  ------------------  ---------------
8.258   0     memory W0 [commit 1-3]  [1, 2, 3, 4, 5, 6]  6              
11.262  0     memory W1 [commit 4-9]  [4, 5, 6, 7, 8, 9]  6              

decoder memory now: [DecoderMemorySnapshot(pool='default', unit=0, capacity_rounds=None, occupied_rounds=0, peak_occupied_rounds=6, admissions=2)]


## 8. Decoder engine: fetch from the unit's memory, algorithm, release

In [10]:
table([r for r in TRACE if r["stage"] == "decoder manager: start decode"], ["t", "window", "unit"])
print()
table([dict(window=f"W{s.window_id}", stage=s.stage, cycles=s.cycles, start=us(s.start_ticks), end=us(s.end_ticks), duration_us=us(s.end_ticks - s.start_ticks)) for s in decoder_engine.stage_records], ["window", "stage", "cycles", "start", "end", "duration_us"])
print()
table([r for r in TRACE if r["stage"] == "decoder engine: result"], ["t", "window", "syndrome_rounds_fetched", "defects", "correction_weight", "logical", "boundary_defects"])

t       window                  unit
------  ----------------------  ----
8.258   memory W0 [commit 1-3]  0   
11.262  memory W1 [commit 4-9]  0   

window  stage      cycles  start   end     duration_us
------  ---------  ------  ------  ------  -----------
W0      fetch      6       8.258   8.282   0.024      
W0      algorithm  None    8.282   8.31    0.028      
W0      release    1       8.31    8.314   0.004      
W1      fetch      6       11.262  11.286  0.024      
W1      algorithm  None    11.286  11.314  0.028      
W1      release    1       11.314  11.318  0.004      

t       window                  syndrome_rounds_fetched  defects  correction_weight  logical  boundary_defects
------  ----------------------  -----------------------  -------  -----------------  -------  ----------------
8.314   memory W0 [commit 1-3]  [1, 2, 3, 4, 5, 6]       6        3                  (1,)     None            
11.318  memory W1 [commit 4-9]  [4, 5, 6, 7, 8, 9]       7        4          

## 9. Boundary to the next window (DD, at decode done), correction to the Pauli frame (WDO)

In [11]:
table([r for r in TRACE if r["stage"] == "boundary handoff (DD) at decode done"], ["t", "window", "to", "boundary"])
print()
dd = [x for x in done.result.link_traffic["transfers"] if x["path"] in ("dd", "wdo")]
table([dict(path=x["path"], window=f"W{x['attribution']['window_id']}", sent=us(x["send_ticks"]), delivered=us(x["delivery_ticks"]), delay_us=us(x["total_delay_ticks"])) for x in dd], ["path", "window", "sent", "delivered", "delay_us"])
print()
table([r for r in TRACE if r["stage"] == "pauli frame: correction arrived (after WDO)"], ["t", "window", "logical"])
print()
table([dict(window=f"W{c.window_key[1]}", accepted=us(c.accepted_ticks), committed=us(c.committed_ticks), logical=c.logical_observables) for c in done.pauli_frame.snapshot().records], ["window", "accepted", "committed", "logical"])

t       window  to      boundary                                                                                                                                            
------  ------  ------  ----------------------------------------------------------------------------------------------------------------------------------------------------
8.314   W0      ['W1']  DependencyResidual(detector_ids=(4, 6, 7, 12, 14), defects={2: [1, 0, 1, 1], 3: [1, 0, 1]})                                                         
11.318  W1      []      DependencyResidual(detector_ids=(36, 44, 53, 56, 60, 63, 71), defects={6: [1], 7: [1], 8: [0, 1, 0, 0, 1], 9: [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1]})

path  window  sent    delivered  delay_us
----  ------  ------  ---------  --------
dd    W0      8.314   8.814      0.5     
wdo   W0      8.314   9.314      1.0     
wdo   W1      11.318  12.318     1.0     

t       window  logical
------  ------  -------
9.314   W0      (1,)   
12.318  W1      (0,)   


## 10. Result of the shot and the whole timeline

In [12]:
r = done.result.operation_results[0]
print("predicted logical observables:", r.logical_observables, "| truth:", r.observable_truth, "| logical failure:", r.logical_failure)
w = sorted(done.window_manager.windows.items())
table([dict(window=f"W{k}", first_round=us(x.t_first_round), data_complete=us(x.t_data_complete), queued=us(x.t_queued), unit_assigned=us(x.t_dispatch), decode_done=us(x.t_done)) for (o, k), x in w], ["window", "first_round", "data_complete", "queued", "unit_assigned", "decode_done"])
print()
print("\n".join(done.engine.log_lines))

predicted logical observables: (1,) | truth: (1,) | logical failure: False
window  first_round  data_complete  queued  unit_assigned  decode_done
------  -----------  -------------  ------  -------------  -----------
W0      1.254        6.258          6.258   6.258          8.314      
W1      4.258        9.262          9.262   9.262          11.318     

[  0.000 us] Controller: START memory  (Clifford, qubits (0,))
[  1.000 us] QPU: memory fires round 1/9
[  1.254 us] DecoderCluster: round 1 of memory arrived (op now has rounds 1..1)
[  2.000 us] QPU: memory fires round 2/9
[  2.258 us] DecoderCluster: round 2 of memory arrived (op now has rounds 1..2)
[  3.000 us] QPU: memory fires round 3/9
[  3.258 us] DecoderCluster: round 3 of memory arrived (op now has rounds 1..3)
[  4.000 us] QPU: memory fires round 4/9
[  4.258 us] DecoderCluster: round 4 of memory arrived (op now has rounds 1..4)
[  5.000 us] QPU: memory fires round 5/9
[  5.258 us] DecoderCluster: round 5 of memory arriv